# BERTopic – Economic Dimension (Optimized)

Optimized BERTopic pipeline for Economic Dimension of Sustainable Fashion.
- No forced number of topics
- English-only filtering
- Economic-focused stopwords
- Interactive bar chart visualization


In [ ]:
!pip -q install bertopic sentence-transformers umap-learn hdbscan openpyxl emoji tqdm langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 33.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from langdetect import detect
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
# Load dataset
DATA_PATH = "/content/data clean economic stage 1.xlsx"
df = pd.read_excel(DATA_PATH)
TEXT_COL = "clean_text"
docs_raw = df[TEXT_COL].dropna().astype(str).tolist()
print("Docs raw:", len(docs_raw))

Docs raw: 7367


In [ ]:
# Cleaning
URL_RE = re.compile(r"http\S+|www\S+")
MULTI_SPACE_RE = re.compile(r"\s+")

def clean_text(s):
    s = s.lower().strip()
    s = URL_RE.sub(" ", s)
    s = emoji.replace_emoji(s, replace=" ")
    s = re.sub(r"[^a-z\s]", " ", s)
    s = MULTI_SPACE_RE.sub(" ", s).strip()
    return s

docs = [clean_text(d) for d in docs_raw]
docs = [d for d in docs if len(d.split()) >= 5]
print("After cleaning:", len(docs))

After cleaning: 6479


In [ ]:
# English-only filter
def filter_english(docs):
    filtered = []
    for d in tqdm(docs):
        try:
            if detect(d) == "en":
                filtered.append(d)
        except:
            continue
    return filtered

docs = filter_english(docs)
print("After EN filter:", len(docs))

  0%|          | 0/6479 [00:00<?, ?it/s]

After EN filter: 6101


In [ ]:
# Stopwords
generic_fillers = {"think","need","make","people","just","like","good","great","also"}
fashion_general = {"fashion","brand","brands","clothing","clothes","trend","collection","sustainable"}
remove_non_economic = {"carbon","emission","waste","pollution","heritage","craft","artisan","cultural"}

stopwords = set(ENGLISH_STOP_WORDS) | generic_fillers | fashion_general | remove_non_economic

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords),
    ngram_range=(1,3),
    min_df=7,
    max_df=0.8
)

In [ ]:
# Model components
embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(n_neighbors=12, n_components=5, min_dist=0.0, metric="cosine", random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=10, metric="euclidean", cluster_selection_method="eom", prediction_data=True)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Train BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()
topic_info

2026-02-26 09:40:12,063 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

2026-02-26 09:51:14,211 - BERTopic - Embedding - Completed ✓
2026-02-26 09:51:14,212 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-26 09:51:58,304 - BERTopic - Dimensionality - Completed ✓
2026-02-26 09:51:58,306 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-26 09:51:59,173 - BERTopic - Cluster - Completed ✓
2026-02-26 09:51:59,190 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 09:51:59,604 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2754,-1_luxury_work_prices_size,"[luxury, work, prices, size, pay, today, makes...",[hate ethical consumption capitalism concept r...
1,0,830,0_amp_sustainability_style_organic,"[amp, sustainability, style, organic, sustaina...",[growing market sustainable textiles reshaping...
2,1,240,1_factories_garment_uses_countries,"[factories, garment, uses, countries, china, e...",[lots places shein dont shop im certain things...
3,2,203,2_videos_video_channel_content,"[videos, video, channel, content, youtube, tha...",[video eye opener reality greenwashing thank j...
4,3,189,3_thrift_thrift stores_stores_second hand,"[thrift, thrift stores, stores, second hand, s...",[donate thrift stores generally get second han...
5,4,156,4_choices_lets_demand_trends,"[choices, lets, demand, trends, planet, buy fa...",[think made great point video think propable e...
6,5,132,5_year_years_buying new_old,"[year, years, buying new, old, bought, worn, a...",[average american buys new items clothing per ...
7,6,126,6_cotton_fabric_polyester_wool,"[cotton, fabric, polyester, wool, organic, ret...",[saying polyester bad dodging issue retailers ...
8,7,110,7_secondhand_recycling_resale_report,"[secondhand, recycling, resale, report, global...",[published latin america secondhand clothing m...
9,8,83,8_shirt_shirts_high quality_tshirt,"[shirt, shirts, high quality, tshirt, quite, q...","[high quality shirts paying per shirt, love go..."


In [ ]:
# Visualization
valid_topics = topic_info[topic_info["Topic"] != -1]["Topic"].tolist()

fig = topic_model.visualize_barchart(
    topics=valid_topics,
    n_words=10,
    title="Economic Topic Words Score"
)

fig.write_html("topic_barchart_Economic.html")
fig.show()

In [ ]:
# Visualisasi peta topik 2D yang interaktif
topic_model.visualize_topics()

In [ ]:
import pandas as pd
import numpy as np

# ============================
# BUILD SUMMARY TABLE
# ============================

# Total dokumen
total_docs = len(topics)

# Hitung jumlah dokumen per topic
topic_counts = pd.Series(topics).value_counts().sort_index()

# Hitung total tanpa noise (-1)
total_wo_noise = total_docs - topic_counts.get(-1, 0)

rows = []

for topic_id, n_doc in topic_counts.items():

    # Persentase dari total dokumen
    percent = (n_doc / total_docs) * 100

    # Ambil keywords top 5
    if topic_id != -1:
        words = topic_model.get_topic(topic_id)
        keywords = ", ".join([w[0] for w in words[:5]])

        # Persentase tanpa noise
        percent_wo_noise = (n_doc / total_wo_noise) * 100 if total_wo_noise > 0 else 0
    else:
        keywords = "Noise / Outliers"
        percent_wo_noise = None

    rows.append({
        "Topic": topic_id,
        "n_doc": n_doc,
        "%": round(percent, 2),
        "keywords": keywords,
        "%_wo_noise": round(percent_wo_noise, 2) if percent_wo_noise is not None else None
    })

summary_table = pd.DataFrame(rows).sort_values("Topic").reset_index(drop=True)

summary_table

,Topic,n_doc,%,keywords,%_wo_noise
0,-1,2754,45.14,Noise / Outliers,NaN
1,0,830,13.60,"amp, sustainability, style, organic, sustainab...",24.80
2,1,240,3.93,"factories, garment, uses, countries, china",7.17
3,2,203,3.33,"videos, video, channel, content, youtube",6.07
4,3,189,3.10,"thrift, thrift stores, stores, second hand, shops",5.65
5,4,156,2.56,"choices, lets, demand, trends, planet",4.66
6,5,132,2.16,"year, years, buying new, old, bought",3.94
7,6,126,2.07,"cotton, fabric, polyester, wool, organic",3.76
8,7,110,1.80,"secondhand, recycling, resale, report, global",3.29
9,8,83,1.36,"shirt, shirts, high quality, tshirt, quite",2.48


In [ ]:
summary_table["%"] = summary_table["%"].astype(str) + "%"
summary_table["%_wo_noise"] = summary_table["%_wo_noise"].astype(str) + "%"

summary_table

,Topic,n_doc,%,keywords,%_wo_noise
0,-1,2754,45.14%,Noise / Outliers,nan%
1,0,830,13.6%,"amp, sustainability, style, organic, sustainab...",24.8%
2,1,240,3.93%,"factories, garment, uses, countries, china",7.17%
3,2,203,3.33%,"videos, video, channel, content, youtube",6.07%
4,3,189,3.1%,"thrift, thrift stores, stores, second hand, shops",5.65%
5,4,156,2.56%,"choices, lets, demand, trends, planet",4.66%
6,5,132,2.16%,"year, years, buying new, old, bought",3.94%
7,6,126,2.07%,"cotton, fabric, polyester, wool, organic",3.76%
8,7,110,1.8%,"secondhand, recycling, resale, report, global",3.29%
9,8,83,1.36%,"shirt, shirts, high quality, tshirt, quite",2.48%


In [ ]:
summary_table.to_excel("bertopic_summary_table.xlsx", index=False)
print("✓ Saved: bertopic_summary_table.xlsx")

✓ Saved: bertopic_summary_table.xlsx


In [ ]:
# ==============================
# SELECT ONLY ECONOMIC CORE TOPICS
# ==============================

selected_topics = [5, 12, 27, 3]

# Ambil semua dokumen yang termasuk 4 topik ini
filtered_docs = []
filtered_topics = []

for doc, topic in zip(docs, topics):
    if topic in selected_topics:
        filtered_docs.append(doc)
        filtered_topics.append(topic)

print("Total docs in selected topics:", len(filtered_docs))


Total docs in selected topics: 514


In [ ]:
# ==============================
# REMAP TOPIC LABELS
# ==============================

topic_mapping = {5:0, 12:1, 27:2, 3:3}

remapped_topics = [topic_mapping[t] for t in filtered_topics]


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Define a new CountVectorizer for the economic_core_model
# Adjust min_df and max_df for a smaller number of "documents" (topics).
# There are 4 topics, so min_df=1 ensures terms in at least one topic are considered.
# max_df=1.0 prevents filtering out terms present in many topics, as the previous max_df (0.8 * 4 = 3.2 documents)
# was causing the error due to being less than min_df.
vectorizer_model_core = CountVectorizer(
    stop_words=list(stopwords), # Reuse the stopwords from the original model
    ngram_range=(1,3),
    min_df=1, # Adjusted from 7
    max_df=1.0 # Adjusted from 0.8
)

# Build new BERTopic model ONLY for selected docs
economic_core_model = BERTopic(
    embedding_model=topic_model.embedding_model,
    vectorizer_model=vectorizer_model_core, # Use the new vectorizer model
    ctfidf_model=topic_model.ctfidf_model,
    verbose=True
)

economic_core_model.fit(filtered_docs, y=remapped_topics)

economic_core_model.get_topic_info()

2026-02-21 09:37:24,919 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

2026-02-21 09:38:10,711 - BERTopic - Embedding - Completed ✓
2026-02-21 09:38:10,712 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-21 09:38:12,352 - BERTopic - Dimensionality - Completed ✓
2026-02-21 09:38:12,354 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-21 09:38:12,378 - BERTopic - Cluster - Completed ✓
2026-02-21 09:38:12,383 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-21 09:38:12,450 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,0,230,0_thrift_buy_stores_hand,"[thrift, buy, stores, hand, second, second han...",[wrt shopping secondhand fast fashion like she...
1,1,166,1_companies_capitalism_world_consumers,"[companies, capitalism, world, consumers, stop...",[shein supply demand people obsessed buying ne...
2,2,87,2_quality_expensive_better_cost,"[quality, expensive, better, cost, mean, payin...",[think comparing lower priced item higher pric...
3,3,31,3_sustainablefashion_traditional_egypt_economic,"[sustainablefashion, traditional, egypt, econo...",[economic achievements egypt traditional cloth...


In [ ]:
fig = economic_core_model.visualize_barchart(
    topics=[0,1,2,3],
    n_words=10,
    title="Economic Core Topics (Reordered)"
)

fig.show()
